![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_CrossEncoder.ipynb)

## Import ONNX CrossEncoder models from HuggingFace 🤗 into Spark NLP 🚀

Let's keep in mind a few things before we start 😊


- `CrossEncoder` is a pairwise BERT sequence classifier: it jointly encodes a `(query, passage)` pair as `[CLS] query [SEP] passage [SEP]` and returns a single relevance score in `[0, 1]` (sigmoid). It is the Spark NLP equivalent of a `sentence-transformers` `CrossEncoder`.
- This annotator supports the ONNX engine only.
- You can import any BERT-family cross-encoder with a single-logit regression head, e.g. `cross-encoder/ms-marco-MiniLM-L6-v2`.

## Export and Save HuggingFace model

- Let's install `transformers` and `optimum` to export the model to ONNX.
- We lock `transformers` to a known good version for reproducibility.

In [1]:
!pip install -q --upgrade transformers optimum[onnxruntime] onnx onnxruntime sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


- HuggingFace's `optimum` exports a standard `AutoModelForSequenceClassification` checkpoint to ONNX directly.
- `cross-encoder/ms-marco-MiniLM-L6-v2` has `num_labels=1` (a regression head), so no custom tracing is needed.

In [2]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
EXPORT_PATH = f"onnx_models/{MODEL_NAME}"

ort_model = ORTModelForSequenceClassification.from_pretrained(MODEL_NAME, export=True)
ort_model.save_pretrained(EXPORT_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(EXPORT_PATH)

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The model cross-encoder/ms-marco-MiniLM-L6-v2 was already converted to ONNX but got `export=True`, the mo

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


('onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/tokenizer_config.json',
 'onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/special_tokens_map.json',
 'onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/vocab.txt',
 'onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/added_tokens.json',
 'onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/tokenizer.json')

Let's have a look inside the export directory:

In [3]:
!ls -l {EXPORT_PATH}

total 89776
-rw-r--r-- 1 root root      765 Jul 12 07:48 config.json
-rw-r--r-- 1 root root 90964641 Jul 12 07:48 model.onnx
-rw-r--r-- 1 root root      695 Jul 12 07:48 special_tokens_map.json
-rw-r--r-- 1 root root     1272 Jul 12 07:48 tokenizer_config.json
-rw-r--r-- 1 root root   711396 Jul 12 07:48 tokenizer.json
-rw-r--r-- 1 root root   231508 Jul 12 07:48 vocab.txt


- We need to move `vocab.txt` into an `assets` folder, which is where Spark NLP looks for tokenizer assets.
- The regression head has no labels, so no `labels.txt` is required.

In [4]:
!mkdir -p {EXPORT_PATH}/assets
!mv {EXPORT_PATH}/vocab.txt {EXPORT_PATH}/assets/

In [5]:
!ls -lR {EXPORT_PATH}

onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2:
total 89552
drwxr-xr-x 2 root root     4096 Jul 12 07:48 assets
-rw-r--r-- 1 root root      765 Jul 12 07:48 config.json
-rw-r--r-- 1 root root 90964641 Jul 12 07:48 model.onnx
-rw-r--r-- 1 root root      695 Jul 12 07:48 special_tokens_map.json
-rw-r--r-- 1 root root     1272 Jul 12 07:48 tokenizer_config.json
-rw-r--r-- 1 root root   711396 Jul 12 07:48 tokenizer.json

onnx_models/cross-encoder/ms-marco-MiniLM-L6-v2/assets:
total 228
-rw-r--r-- 1 root root 231508 Jul 12 07:48 vocab.txt


Voila! We have our `vocab.txt` inside the `assets` folder and the `model.onnx` at the export root.

## Import and Save CrossEncoder in Spark NLP

- Let's install and set up Spark NLP in your environment. Make sure the version you install includes `CrossEncoder`.

In [ ]:
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

Let's start a Spark NLP session:

In [ ]:
import sparknlp

spark = sparknlp.start()
print(sparknlp.version())

- Let's use `loadSavedModel`, which takes the export folder and a `SparkSession`.
- The two input columns are the two `DOCUMENT` columns of the pair (query and passage).

In [ ]:
from sparknlp.annotator import CrossEncoder

crossEncoder = CrossEncoder.loadSavedModel(EXPORT_PATH, spark) \
    .setInputCols(["document1", "document2"]) \
    .setOutputCol("score") \
    .setCaseSensitive(False)

- Let's save it on disk so it is easier to be moved around and reused later.

In [ ]:
crossEncoder.write().overwrite().save(f"{MODEL_NAME}_spark_nlp_onnx")

Let's clean up the export folder we no longer need:

In [ ]:
!rm -rf {EXPORT_PATH}

Now let's load the saved Spark NLP model back:

In [ ]:
crossEncoder_loaded = CrossEncoder.load(f"{MODEL_NAME}_spark_nlp_onnx") \
    .setInputCols(["document1", "document2"]) \
    .setOutputCol("score")

This is how you use it for reranking. A `MultiDocumentAssembler` builds the two `DOCUMENT` columns, one query duplicated against several passages (a layout you build upstream with a `crossJoin`/`explode`).

In [ ]:
from sparknlp.base import MultiDocumentAssembler
from pyspark.ml import Pipeline

document = MultiDocumentAssembler() \
    .setInputCols(["query", "passage"]) \
    .setOutputCols(["document1", "document2"])

pipeline = Pipeline(stages=[document, crossEncoder_loaded])

query = "How many people live in Berlin?"
passages = [
    "Berlin has a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin is well known for its museums.",
    "In 2014, the city state Berlin had 37,368 live births.",
]

data = spark.createDataFrame([[query, p] for p in passages]).toDF("query", "passage")
result = pipeline.fit(data).transform(data)
result.select("passage", "score.result").show(truncate=80)

That's it! You can now go wild and use hundreds of `CrossEncoder` models as rerankers in Spark NLP 🚀